# Análise Profunda da Função `build_fold_predicate_graph`

## 📊 O Que a Função Constrói

A função constrói um **grafo direcionado acíclico (DAG)** que representa **caminhos de decisão** baseados em predicados espectrais, ordenados por importância (Mutual Information ou Covariância).

---

## 🏗️ Estrutura do Grafo

### **Nós (Vértices)**

1. **Nós Predicados**: Regras sobre zonas espectrais
   - Exemplo: `"Ca ka <= 25.5"`, `"Fe ka > 10.2"`
   - Tipo: `node_type='predicate'`

2. **Nós Terminais**: Classes finais de classificação
   - `'Class_A'` e `'Class_B'`
   - Tipo: `node_type='terminal'`
   - Atributo adicional: `class_label='A'` ou `'B'`

### **Arestas (Conexões Direcionadas)**

As arestas representam **transições sequenciais** entre predicados, seguindo a ordem de importância:



In [ ]:
P1 (mais importante) → P2 → P3 → ... → Pn (menos importante) → Class_X



---

## 🔄 Como os Caminhos São Construídos

### **Processamento por Fold (Bag)**

Para cada fold no `bags_result`:

1. **Ordenação por Importância**:
   - Predicados são ordenados pelo `mi_results_dict` (decrescente)
   - Rank 1 = maior MI/Cov (mais importante)
   - Rank N = menor MI/Cov (menos importante)

2. **Criação do Caminho Sequencial**:
   ```
   Fold_1: P5 → P12 → P3 → P8 → Class_A
   Fold_2: P12 → P5 → P21 → P3 → Class_B
   Fold_3: P8 → P3 → P12 → Class_A
   ```

3. **Acumulação de Arestas**:
   - Se a aresta já existe (ex: P5→P12 aparece em múltiplos folds), o comportamento depende do **modo de peso**

---

## ⚖️ Modos de Peso nas Arestas

### **1. Modo `'ranking'` (Padrão)**

#### **Lógica do Peso**
- **Peso = Posição invertida no ranking**
- Predicados mais importantes (rank 1) contribuem com **maior peso**

#### **Fórmulas**

##### **Sem Normalização** (`normalize_weights=False`)


In [ ]:
peso = (n_predicados - rank + 1)

- **Exemplo** (fold com 5 predicados):
  - Rank 1 → peso = 5
  - Rank 2 → peso = 4
  - Rank 3 → peso = 3
  - Rank 4 → peso = 2
  - Rank 5 → peso = 1 (último → terminal)

##### **Com Normalização** (`normalize_weights=True`)


In [ ]:
peso = (n_predicados - rank) / n_predicados

- **Exemplo** (fold com 5 predicados):
  - Rank 1 → peso = 0.8
  - Rank 2 → peso = 0.6
  - Rank 3 → peso = 0.4
  - Rank 4 → peso = 0.2
  - Rank 5 → peso = 0.0 (último → terminal)

#### **Acumulação de Pesos**
- **Se mesma aresta aparece em múltiplos folds: SOMA os pesos**

**Exemplo Prático**:


In [ ]:
Fold_1: P5 (rank 1, 5 preds) → P12 (rank 2)  ➔  peso = 5
Fold_2: P5 (rank 2, 4 preds) → P12 (rank 3)  ➔  peso = 3
Fold_3: P5 (rank 1, 6 preds) → P12 (rank 2)  ➔  peso = 6

Peso final da aresta P5→P12 = 5 + 3 + 6 = 14



#### **Desempate Bidirecional**
- Se existe P5→P12 e P12→P5:
  - **Mantém a aresta com MAIOR peso acumulado**
  - Remove a aresta perdedora

---

### **2. Modo `'cooccurrence'`**

#### **Lógica do Peso**
- **Peso = Número de amostras que satisfazem AMBOS os predicados**
- Obtido da matriz de co-ocorrência (simétrica)

#### **Fórmula**


In [ ]:
peso = co_occurrence_matrix[pred_origem, pred_destino]



**Exemplo Prático**:


In [ ]:
- 100 amostras satisfazem "Ca ka <= 25.5"
- 80 amostras satisfazem "Fe ka > 10.2"
- 60 amostras satisfazem AMBOS

➔ Peso da aresta "Ca ka <= 25.5" → "Fe ka > 10.2" = 60



#### **Acumulação de Pesos**
- **Se mesma aresta aparece em múltiplos folds: NÃO soma (mantém peso original)**
- A matriz de co-ocorrência já representa todas as amostras

#### **Score de Confiança Paralelo**
Como a matriz é simétrica (`cooc[A,B] = cooc[B,A]`), o peso não pode resolver arestas bidirecionais. Por isso, calcula-se um **score de confiança** paralelo:



In [ ]:
Score(A→B) = Σ (n_predicados - rank_A + 1) para cada fold onde A→B aparece



**Exemplo**:


In [ ]:
Fold_1: P5 (rank 1, 5 preds) → P12 (rank 2)  ➔  score += 5
Fold_2: P5 (rank 3, 4 preds) → P12 (rank 4)  ➔  score += 2
Fold_3: P5 (rank 1, 6 preds) → P12 (rank 2)  ➔  score += 6

Score de confiança P5→P12 = 5 + 2 + 6 = 13



#### **Desempate Bidirecional**
- Se existe P5→P12 e P12→P5:
  - **Mantém a aresta com MAIOR score de confiança**
  - Remove a aresta perdedora

#### **Multiplicador de Confiança** (`apply_confidence_multiplier=True`)
Após resolver bidirecionais, **multiplica o peso pelo score**:



In [ ]:
peso_final = co_ocorrência × score_de_confiança



**Exemplo**:


In [ ]:
Aresta: P5 → P12
- Peso (co-ocorrência): 60
- Score de confiança: 13

Com multiplicador:
- Peso final = 60 × 13 = 780



**Interpretação**: Combina **magnitude das amostras** (co-ocorrência) com **consistência de importância** (score).

---

## 🔗 Conexão ao Terminal

### **Último Predicado → Nó Terminal**

Após construir o caminho sequencial, o **último predicado** (menor importância) se conecta ao nó terminal:

1. **Determina a classe majoritária**:
   - Conta quantas amostras do predicado pertencem a cada classe predita
   - Escolhe a classe com mais amostras

2. **Cria aresta para o terminal correspondente**:
   ```
   ... → Pn → Class_A  (se maioria for classe A)
   ... → Pm → Class_B  (se maioria for classe B)
   ```

### **Peso da Aresta Terminal**

#### **Modo `'ranking'`**
- **Sem normalização**: `peso = 1` (score mínimo)
- **Com normalização**: `peso = 0.0`

#### **Modo `'cooccurrence'`**
- `peso = len(df_last)` (número de amostras do último predicado)

---

## 📝 Exemplo Completo

### **Configuração**


In [ ]:
build_fold_predicate_graph(
    bags_result=bags,
    mi_results_dict=mi_results,
    predicates_df=predicates,
    weight_mode='ranking',
    normalize_weights=False
)



### **Dados de Entrada**

**Fold_1** (5 predicados):


In [ ]:
MI Ranking:
1. "Ca ka <= 25.5"  (MI=0.82)
2. "Fe ka > 10.2"   (MI=0.71)
3. "Ti ka <= 5.0"   (MI=0.65)
4. "Si ka > 15.0"   (MI=0.52)
5. "Al ka <= 8.0"   (MI=0.38)

Classe majoritária do último: A



**Fold_2** (4 predicados):


In [ ]:
MI Ranking:
1. "Fe ka > 10.2"   (MI=0.79)
2. "Ca ka <= 25.5"  (MI=0.73)
3. "Si ka > 15.0"   (MI=0.60)
4. "Ti ka <= 5.0"   (MI=0.45)

Classe majoritária do último: B



### **Construção do Grafo**

#### **Fold_1: Arestas e Pesos**


In [ ]:
"Ca ka <= 25.5" → "Fe ka > 10.2"   | peso=5 (rank 1, n=5)
"Fe ka > 10.2"  → "Ti ka <= 5.0"   | peso=4 (rank 2, n=5)
"Ti ka <= 5.0"  → "Si ka > 15.0"   | peso=3 (rank 3, n=5)
"Si ka > 15.0"  → "Al ka <= 8.0"   | peso=2 (rank 4, n=5)
"Al ka <= 8.0"  → Class_A          | peso=1 (último → terminal)



#### **Fold_2: Arestas e Pesos**


In [ ]:
"Fe ka > 10.2"  → "Ca ka <= 25.5"  | peso=4 (rank 1, n=4)
"Ca ka <= 25.5" → "Si ka > 15.0"   | peso=3 (rank 2, n=4)
"Si ka > 15.0"  → "Ti ka <= 5.0"   | peso=2 (rank 3, n=4)
"Ti ka <= 5.0"  → Class_B          | peso=1 (último → terminal)



#### **Acumulação de Pesos**


In [ ]:
"Ca ka <= 25.5" → "Fe ka > 10.2"   | peso=5       (só Fold_1)
"Fe ka > 10.2"  → "Ca ka <= 25.5"  | peso=4       (só Fold_2)
"Fe ka > 10.2"  → "Ti ka <= 5.0"   | peso=4       (Fold_1)
"Ti ka <= 5.0"  → "Si ka > 15.0"   | peso=3       (Fold_1)
"Ca ka <= 25.5" → "Si ka > 15.0"   | peso=3       (só Fold_2)
"Si ka > 15.0"  → "Al ka <= 8.0"   | peso=2       (só Fold_1)
"Si ka > 15.0"  → "Ti ka <= 5.0"   | peso=2       (só Fold_2)



#### **Resolução de Bidirecionais**


In [ ]:
Par encontrado: "Ca ka <= 25.5" ↔ "Fe ka > 10.2"
- Ca→Fe: peso=5
- Fe→Ca: peso=4
➔ Remove Fe→Ca (peso menor)
➔ Mantém Ca→Fe

Par encontrado: "Fe ka > 10.2" ↔ "Ti ka <= 5.0"
- Fe→Ti: peso=4
- Ti→Fe: Não existe
➔ Nenhuma remoção necessária

Par encontrado: "Ti ka <= 5.0" ↔ "Si ka > 15.0"
- Ti→Si: peso=3
- Si→Ti: peso=2
➔ Remove Si→Ti (peso menor)
➔ Mantém Ti→Si



### **Grafo Final**


In [ ]:
Nós: 7 (5 predicados + 2 terminais)
Arestas: 7 (após remoção de 2 bidirecionais)

Estrutura:
"Ca ka <= 25.5" → "Fe ka > 10.2" (peso=5)
"Fe ka > 10.2"  → "Ti ka <= 5.0" (peso=4)
"Ti ka <= 5.0"  → "Si ka > 15.0" (peso=3)
"Ca ka <= 25.5" → "Si ka > 15.0" (peso=3)
"Si ka > 15.0"  → "Al ka <= 8.0" (peso=2)
"Al ka <= 8.0"  → Class_A        (peso=1)
"Ti ka <= 5.0"  → Class_B        (peso=1)



---

## 🎯 Resumo das Configurações

| Parâmetro | Opção | Efeito |
|-----------|-------|--------|
| `weight_mode` | `'ranking'` | Peso = rank invertido (importância) |
| | `'cooccurrence'` | Peso = co-ocorrência (amostras) |
| `normalize_weights` | `True` | Normaliza pesos [0, 1] (só ranking) |
| | `False` | Pesos inteiros [1, k] |
| `apply_confidence_multiplier` | `True` | Multiplica cooc × score (só cooccurrence) |
| | `False` | Usa apenas co-ocorrência |

**Resultado**: Grafo que representa **caminhos de decisão** ordenados por importância, com pesos refletindo **relevância** (ranking) ou **cobertura** (co-ocorrência).